WDICSV.csv contains 1,500+ indicators across 200+ countries, which made it too large for Excel to sort and filter reliably, as  it kept crashing. This step was moved to Python, which handled sorting, filtering, cleaning, and merging a file this size without issue.

1. Load both sorce files first

WDICSV.csv = the full inidicator data

WDICou try,csv = country metada (Region, Income Group)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Step 1: Load both source files first
wdi = pd.read_csv('WDICSV.csv', engine='python', on_bad_lines='skip')
country = pd.read_csv('WDICountry.csv', engine='python', on_bad_lines='skip')

print(f"WDICSV rows loaded: {len(wdi)}")
print(f"WDICountry rows loaded: {len(country)}")

WDICSV rows loaded: 396970
WDICountry rows loaded: 264


2. Define the 6 indicators chosen from the series csv file

Theme: Environment > Density & Urbanization

In [ ]:
indicator_codes = [
    'EN.POP.DNST',        # Population density (people per sq. km)
    'SP.URB.TOTL.IN.ZS',  # Urban population (% of total)
    'SP.URB.GROW',        # Urban population growth (annual %)
    'EN.URB.MCTY.TL.ZS',  # Population in cities >1M (% of total)
    'EN.POP.SLUM.UR.ZS',  # Slum population (% of urban)
    'SP.RUR.TOTL.ZS'      # Rural population (% of total)
]

3. Filtered WDICSV down to these 6 inidicators. This was the step that kept crashig in excel due to the size of the file

In [ ]:
wdi_filtered = wdi[wdi['Indicator Code'].isin(indicator_codes)].copy()
print("Filtered rows:", len(wdi_filtered))

Filtered rows: 1590


4. Trimming the dataset to 2000-2024 only. This allows for a more uo to date comparison

In [ ]:
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']
year_cols = [str(y) for y in range(2000, 2025)]
wdi_trimmed = wdi_filtered[id_cols + year_cols]


5. Reshaping of table form one column per year to one row per country.

In [ ]:
wdi_long = wdi_trimmed.melt(
    id_vars=id_cols,
    var_name='Year',
    value_name='Value'
)
wdi_long['Year'] = wdi_long['Year'].astype(int)


6. Handling missing values, of few selected cells in order not to loose value.

In [ ]:
strong_indicators = ['EN.POP.DNST', 'SP.URB.TOTL.IN.ZS', 'SP.URB.GROW', 'SP.RUR.TOTL.ZS']
mask_strong = wdi_long['Indicator Code'].isin(strong_indicators)
wdi_long = wdi_long[~(mask_strong & wdi_long['Value'].isna())]

7. Merging the region and income group cells in wdi country

In [ ]:
country_meta = country[['Country Code', 'Region', 'Income Group']]
merged = wdi_long.merge(country_meta, on='Country Code', how='left')


8. Seperate individual countries from world bank aggregates

In [ ]:
aggregates = merged[merged['Region'].isna()]
countries_only = merged[merged['Region'].notna()]

print("Individual country rows:", len(countries_only))
print("Aggregate rows (World, regions, income groups):", len(aggregates))


Individual country rows: 32237
Aggregate rows (World, regions, income groups): 7053


In [ ]:
num_duplicates = countries_only.duplicated().sum()
print(f"Number of duplicated rows in countries_only: {num_duplicates}")

if num_duplicates > 0:
    print("Displaying duplicated rows:")
    display(countries_only[countries_only.duplicated(keep=False)])

Number of duplicated rows in countries_only: 0


9. Duplicate check

In [19]:
print(countries_only.info())
print(countries_only.isna().sum())
print(countries_only['Indicator Code'].value_counts())

<class 'pandas.core.frame.DataFrame'>
Index: 32237 entries, 284 to 39289
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    32237 non-null  object 
 1   Country Code    32237 non-null  object 
 2   Indicator Name  32237 non-null  object 
 3   Indicator Code  32237 non-null  object 
 4   Year            32237 non-null  int64  
 5   Value           26427 non-null  float64
 6   Region          32237 non-null  object 
 7   Income Group    32237 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 2.2+ MB
None
Country Name         0
Country Code         0
Indicator Name       0
Indicator Code       0
Year                 0
Value             5810
Region               0
Income Group         0
dtype: int64
Indicator Code
EN.URB.MCTY.TL.ZS    5425
EN.POP.SLUM.UR.ZS    5425
SP.URB.TOTL.IN.ZS    5425
SP.RUR.TOTL.ZS       5425
SP.URB.GROW          5424
EN.POP.DNST          5113
Name: coun

10. Checking for Inconsistencies: Unique Values in Categorical Columns



In [ ]:
print('Unique Country Names:', countries_only['Country Name'].nunique())
print('Unique Country Codes:', countries_only['Country Code'].nunique())
print('Unique Indicator Names:', countries_only['Indicator Name'].nunique())
print('Unique Indicator Codes:', countries_only['Indicator Code'].nunique())
print('Unique Regions:', countries_only['Region'].nunique())
print('Unique Income Groups:', countries_only['Income Group'].nunique())

# Displaying the unique values for smaller columns to visually inspect
print('\nRegions:\n', countries_only['Region'].unique())
print('\nIncome Groups:\n', countries_only['Income Group'].unique())
print('\nIndicator Codes (first 10):\n', countries_only['Indicator Code'].unique()[:10])

Unique Country Names: 217
Unique Country Codes: 217
Unique Indicator Names: 6
Unique Indicator Codes: 6
Unique Regions: 7
Unique Income Groups: 4

Regions:
 ['Middle East & North Africa' 'Europe & Central Asia'
 'East Asia & Pacific' 'Sub-Saharan Africa' 'Latin America & Caribbean'
 'South Asia' 'North America']

Income Groups:
 ['Low income' 'Upper middle income' 'High income' 'Lower middle income']

Indicator Codes (first 10):
 ['EN.POP.DNST' 'EN.URB.MCTY.TL.ZS' 'EN.POP.SLUM.UR.ZS' 'SP.RUR.TOTL.ZS'
 'SP.URB.TOTL.IN.ZS' 'SP.URB.GROW']


11. descriptive statistics

In [ ]:
print('\nYear range:')
print(f"Min Year: {countries_only['Year'].min()}, Max Year: {countries_only['Year'].max()}")

print('\nValue column descriptive statistics:')
display(countries_only['Value'].describe())


Year range:
Min Year: 2000, Max Year: 2024

Value column descriptive statistics:


,Value
count,26427.000000
mean,105.603387
std,841.207149
min,-10.885479
25%,5.410697
50%,32.292868
75%,66.696986
max,21530.000000


12. Exporting and loading  files

In [ ]:
countries_only.to_csv('wdi_urban_density_clean.csv', index=False)
aggregates.to_csv('wdi_urban_density_aggregates.csv', index=False)

In [ ]:
# Load the cleaned CSV back into a DataFrame
final_df = pd.read_csv('wdi_urban_density_clean.csv')

# Display the first 5 rows of the DataFrame
display(final_df)

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value,Region,Income Group
0,Afghanistan,AFG,Population density (people per sq. km of land ...,EN.POP.DNST,2000,30.863847,Middle East & North Africa,Low income
1,Afghanistan,AFG,Population in urban agglomerations of more tha...,EN.URB.MCTY.TL.ZS,2000,11.927819,Middle East & North Africa,Low income
2,Afghanistan,AFG,Population living in slums (% of urban populat...,EN.POP.SLUM.UR.ZS,2000,NaN,Middle East & North Africa,Low income
3,Afghanistan,AFG,Rural population (% of total population),SP.RUR.TOTL.ZS,2000,81.441800,Middle East & North Africa,Low income
4,Afghanistan,AFG,Urban population (% of total population),SP.URB.TOTL.IN.ZS,2000,18.558200,Middle East & North Africa,Low income
...,...,...,...,...,...,...,...,...
32232,Zimbabwe,ZWE,Population in urban agglomerations of more tha...,EN.URB.MCTY.TL.ZS,2024,9.637881,Sub-Saharan Africa,Lower middle income
32233,Zimbabwe,ZWE,Population living in slums (% of urban populat...,EN.POP.SLUM.UR.ZS,2024,NaN,Sub-Saharan Africa,Lower middle income
32234,Zimbabwe,ZWE,Rural population (% of total population),SP.RUR.TOTL.ZS,2024,60.106628,Sub-Saharan Africa,Lower middle income
32235,Zimbabwe,ZWE,Urban population (% of total population),SP.URB.TOTL.IN.ZS,2024,39.893372,Sub-Saharan Africa,Lower middle income


In [20]:
# Load the aggregate CSV back into a DataFrame
aggregates_df = pd.read_csv('wdi_urban_density_aggregates.csv')

# Display the first 5 rows of the aggregates DataFrame
display(aggregates_df)

,Country Name,Country Code,Indicator Name,Indicator Code,Year,Value,Region,Income Group
0,Africa Eastern and Southern,AFE,Population density (people per sq. km of land ...,EN.POP.DNST,2000,33.303511,NaN,NaN
1,Africa Eastern and Southern,AFE,Population in urban agglomerations of more tha...,EN.URB.MCTY.TL.ZS,2000,11.815898,NaN,NaN
2,Africa Eastern and Southern,AFE,Population living in slums (% of urban populat...,EN.POP.SLUM.UR.ZS,2000,53.074562,NaN,NaN
3,Africa Eastern and Southern,AFE,Rural population (% of total population),SP.RUR.TOTL.ZS,2000,71.531312,NaN,NaN
4,Africa Eastern and Southern,AFE,Urban population (% of total population),SP.URB.TOTL.IN.ZS,2000,28.468688,NaN,NaN
...,...,...,...,...,...,...,...,...
7048,World,WLD,Population in urban agglomerations of more tha...,EN.URB.MCTY.TL.ZS,2024,25.074564,NaN,NaN
7049,World,WLD,Population living in slums (% of urban populat...,EN.POP.SLUM.UR.ZS,2024,NaN,NaN,NaN
7050,World,WLD,Rural population (% of total population),SP.RUR.TOTL.ZS,2024,42.434400,NaN,NaN
7051,World,WLD,Urban population (% of total population),SP.URB.TOTL.IN.ZS,2024,57.565600,NaN,NaN


13. Downloading files

In [ ]:
from google.colab import files

# Download the cleaned individual countries data
files.download('wdi_urban_density_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Download the aggregates data
files.download('wdi_urban_density_aggregates.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>